# OFC 2026 | COSMOS Transfer Learning 오류 원인 분석 보고서

**보고서 ID:** OFC26-COSMOS-ERR-001 · **작성일:** 2026-11-02 · **대상:** PI-EDFA / Transformer · **증거 상태:** 실측 학습 로그 + GitHub 비교 JSON

> 목적: COSMOS 사전학습 후 최신 Train 미세조정에서 발생한 성능 저하를 수치로 기록하고, **확인된 소프트웨어 오류**, **측정된 이상 현상**, **검증되지 않은 원인 가설**을 분리한다. 최초 실행 결과는 변경하지 않고 보존한다.

**학습 코드:** [train_transfer.py](https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/train_transfer.py) · **최초 실험 원본:** [pilot benchmark](https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/benchmarks/pilot) · **동일 학습률 대조실험:** [lr_control](https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/benchmarks/lr_control) · **Actions 로그:** [첫 실험](https://github.com/kimheeseo/LSCNS/actions/runs/41358039001), [학습률 대조](https://github.com/kimheeseo/LSCNS/actions/runs/41358641951)

**주의:** 이 보고서의 수치는 공식 공개 데이터를 이용한 2026-11-02 사후 실험의 *내부 Train 홀드아웃* 결과다. 대회 비공개 리더보드 점수와 동일하지 않다.

## 1. Executive Summary

초기 COSMOS 4,000개 샘플 사전학습 후 두 모델의 최근 Train 검증 성능이, 같은 최근 Train만으로 학습한 Scratch 대비 현저히 악화되었다.

| 모델 | Transfer (최고 checkpoint) | Scratch (최고 checkpoint) | Transfer − Scratch |
|---|---:|---:|---:|
| **PI-EDFA** | **3.622148** | **0.226496** | **+3.395652** |
| **Transformer** | **3.733105** | **0.293547** | **+3.439558** |

두 모델 모두 SHB 일부 검증 샘플에서 약 **29 dB의 활성 채널 최대 절대오차**가 발생했고, 공식형 점수의 extreme-error 페널티가 크게 작동했다.

**확정된 코드 오류:** 최초 요약 `transfer_val` / `scratch_val` 필드는 마지막 epoch 점수를 기록했지만, 예측과 Category별 결과는 최고 checkpoint를 사용했다. 예를 들어 Transformer Scratch의 최초 표에는 **0.334849**, 최고 checkpoint의 정확한 비교 점수는 **0.293547**이었다. 현재 학습 코드의 보고 로직을 수정했다.

**원인 판정 주의:** 최초 Transfer 학습률(1e-4)과 Scratch 학습률(2e-3)의 차이, COSMOS와 최신 데이터 분포 차이 가능성, SHB 특이 샘플이 동시에 존재한다. 단일 1차 실험만으로 특정 원인을 확정할 수 없다.

## 2. 재현 실험 조건 및 데이터 경계

| 항목 | 설정 |
|---|---|
| Source | COSMOS **4,000개 학습** / **500개 검증** |
| Target | 공식 Train **397개 학습** / **93개 검증** (총 490행) |
| 유효 검증 채널 | 1,985개 |
| 시드 | 2026 |
| 모델 | PI-EDFA (hidden 96), Transformer (hidden 64) |
| Source 사전학습 | 최대 6 epochs, AdamW, LR=0.001 |
| Target 미세조정(첫 실험) | 최대 55 epochs, LR=0.0001 |
| Scratch(첫 실험) | 최대 55 epochs, LR=0.002 |
| 정규화 | Transfer는 COSMOS Train에 fit한 `StandardScaler`를 Target에서도 재사용 |
| 평가 범위 | Active/WSS ON 채널만 |
| 공개 TestGround | **사용하지 않음** (학습/검증/최적화 모두 제외) |

두 모델은 같은 Target 홀드아웃 인덱스를 사용한다. 다만 첫 실험의 Transfer와 Scratch는 학습률이 다르므로, **COSMOS 가중치의 단독 효과를 분리한 공정한 인과 비교는 아니다.**

In [ ]:
import json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = "https://raw.githubusercontent.com/kimheeseo/LSCNS/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/"
def fetch_json(path):
    with urllib.request.urlopen(ROOT + path, timeout=25) as response:
        return json.load(response)
def fetch_csv(path):
    return pd.read_csv(ROOT + path)

# Stored original measurements, not simulated predictions.
pilot = fetch_json("benchmarks/pilot/comparison.json")
try:
    lr_control = fetch_json("benchmarks/lr_control/comparison.json")
    print("동일 학습률 대조실험 데이터 로드 완료")
except Exception as exc:
    lr_control = None
    print("동일 학습률 대조실험 아직 없음:", str(exc)[:200])
print("Models:", list(pilot), "Control ready:", lr_control is not None)


## 3. 문제 A — 최고 체크포인트와 마지막 Epoch 점수 불일치

**분류: CONFIRMED SOFTWARE REPORTING BUG (수정 완료)**

기존 `train_stage`는 검증 점수가 최저인 `best_state`를 복원한 뒤, 마지막 epoch에서 만들어 둔 `metrics`를 반환했다. 그 결과 `results.json`의 단일 score 필드와 `*_val_by_category.ALL` score가 달랐다.

- PI-EDFA Scratch: 마지막 epoch **0.234593** vs 복원 checkpoint **0.226496**
- Transformer Transfer: 마지막 epoch **3.749114** vs 복원 checkpoint **3.733105**
- Transformer Scratch: 마지막 epoch **0.334849** vs 복원 checkpoint **0.293547**
- Transformer COSMOS 검증: 마지막 epoch **0.207240** vs 학습 로그상 최적 score **0.202472**

**수정 방법:** `net.load_state_dict(best_state)` 이후 동일한 검증 로더로 `selected_metrics = evaluate_loader(net, va, module, device)`를 다시 계산하고, `selected_metrics`를 반환하도록 수정했다. 기존 pilot 원본 JSON은 감사 추적을 위해 변경하지 않는다.

In [ ]:
rows=[]
for model, result in pilot.items():
    for stage in ["transfer", "scratch"]:
        stored=result[f"{stage}_val"]["score"]
        selected=result[f"{stage}_val_by_category"]["ALL"]["score"]
        rows.append({"model":model,"stage":stage,
                     "original_json_last_epoch":stored,
                     "restored_best_checkpoint":selected,
                     "difference":stored-selected})
display(pd.DataFrame(rows).round(6))
assert all(row["restored_best_checkpoint"] <= row["original_json_last_epoch"]+1e-5 for row in rows)


## 4. 문제 B — SHB에서 극단적 Gain 예측 오차

**분류: CONFIRMED OBSERVATION, ROOT CAUSE UNDETERMINED**

첫 실험 최근 Train 홀드아웃의 SHB 구간은 **18개 샘플, 484개 활성 채널**이었다. Transfer 모델에서 SHB의 점수가 급등했고 약 29 dB 최대 오차가 확인됐다.

| 모델 | Aging | SHB | Unseen | 전체 |
|---|---:|---:|---:|---:|
| PI-EDFA Transfer | 0.275386 | **10.765397** | 0.348309 | 3.622148 |
| PI-EDFA Scratch | 0.146664 | 0.158152 | 0.287564 | 0.226496 |
| Transformer Transfer | 0.219533 | **11.159749** | 0.317740 | 3.733105 |
| Transformer Scratch | 0.201646 | 0.286411 | 0.315675 | 0.293547 |

OFC 평가식(이 코드의 구현 기준):

\[
S = \overline{MAE} + 0.3\max(0,P_{95}-\overline{MAE}-0.5) + 0.1\max(0,E_{max}-P_{95}-0.7) + 0.15\overline{STD}.
\]

PI-EDFA Transfer의 전체 점수 **3.622148** 중 약 **2.726154**는 최대오차 페널티 `0.1 × (28.862221 − 0.900684 − 0.7)`에서 발생한다. 이것은 **오차 폭증이 점수를 지배한다는 계산상 확정 사실**이다. 다만 잘못된 입력 스케일, 샘플 도메인 차이, 최적화 부족 중 어느 것이 가장 큰 원인인지는 추가 진단이 필요하다.

In [ ]:
s = []
for m, result in pilot.items():
    for stage in ["transfer", "scratch"]:
        met = result[f"{stage}_val_by_category"]["ALL"]
        s.append({"model":m,"stage":stage,
                  "mae":met["mae"],
                  "std_component":0.15*met["std"],
                  "p95_component":0.3*met["penalty_p95"],
                  "max_component":0.1*met["penalty_max"],
                  "score":met["score"]})
score_parts=pd.DataFrame(s)
display(score_parts.round(6))
np.testing.assert_allclose(
    score_parts[["mae","std_component","p95_component","max_component"]].sum(axis=1),
    score_parts["score"], atol=2e-5)

fig, ax=plt.subplots(figsize=(9,4))
p=score_parts.set_index(["model","stage"])
p[["mae","std_component","p95_component","max_component"]].plot.bar(
    stacked=True, ax=ax)
ax.set_title("Score decomposition — restored best checkpoints")
ax.set_ylabel("Score (lower better)")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()


## 5. 문제 C — 학습률 비교가 달랐다는 실험 설계의 혼란 변수

**분류: CONFIRMED CONFIGURATION CONFOUND**

첫 실험에서 Transfer fine-tune은 `1e-4`, Scratch는 `2e-3`으로, **Scratch 학습률이 20배 컸다.** 55 epochs로 고정한 조건에서 Transfer가 최근 도메인에 충분히 수렴하지 않았을 가능성이 있다.

검증을 위해 사전학습 COSMOS 샘플(4,000), 최대 epoch(6/55), Target 분할, 시드를 유지하고, **Transfer와 Scratch를 모두 LR=0.002**로 맞춘 별도 GitHub Actions 실험을 추가했다. 아래 셀은 그 결과가 GitHub에 게시된 경우 자동으로 비교한다.

> 해석: 학습률을 같게 했을 때 Transfer 성능이 개선되면 '학습률 부족' 가설의 근거는 강화된다. 그러나 그 자체로 전체 COSMOS 전이학습의 우수성이 입증되는 것은 아니다.

In [ ]:
if lr_control is None:
    print("동일 학습률 결과가 아직 게시되지 않았습니다. GitHub Actions 완료 후 재실행하세요.")
else:
    tables=[]
    for name, data in [("pilot_unequal_LR",pilot),("lr_control_equal_LR",lr_control)]:
        for model, d in data.items():
            for stage in ["transfer","scratch"]:
                val=d[f"{stage}_val_by_category"]["ALL"]
                tables.append({"experiment":name,"model":model,"stage":stage,
                               "score":val["score"],"MAE":val["mae"],
                               "max_error_db":val["max_error"],
                               "shb_score":d[f"{stage}_val_by_category"]["shb"]["score"]})
    comparison=pd.DataFrame(tables)
    display(comparison.round(6))
    f,ax=plt.subplots(figsize=(9,4))
    for model in ["pi-edfa","transformer"]:
        sub=comparison[comparison.model==model]
        for stage in ["transfer","scratch"]:
            b=sub[sub.stage==stage]
            ax.plot(b.experiment,b.score,marker="o",label=model+" "+stage)
    ax.set_yscale("log");ax.set_ylabel("Best checkpoint validation score (log)")
    ax.set_title("Controlled LR=0.002 vs initial LR mismatch")
    ax.grid(alpha=.25);ax.legend();plt.tight_layout();plt.show()


## 6. 문제 D — COSMOS/최근 Train 입력 분포 차이 및 SHB 특정 샘플

**분류: INVESTIGATION / HYPOTHESIS**

모델 코드에서는 source(COSMOS) 학습 부분에 `StandardScaler.fit()`을 수행하고 그 평균·표준편차로 **최신 Train까지 transform**한다. 이는 가중치 입력 좌표를 일치시키는 올바른 방식이지만, 두 데이터 분포가 많이 다르면 Target의 `|z|`가 극단적으로 커질 수 있다.

**확인 절차:** 10개 글로벌 입력의 Target 검증 `median|z|`, `p95|z|`, `max|z|`, `|z|>5` 비율을 비교한다. SHB에서 오류가 큰 관측의 **ID, target_gain, target_tilt, 활성 채널 수, 최악 채널 번호, 실측·예측 Gain**을 원본과 대조한다. `|z|>5` 자체는 오류 증거가 아니라 조사 우선순위를 높이는 징후다.

**특히 검증할 순서:** (1) 활성 mask와 실제 label 매칭, (2) channel index 0–94 및 단위(dBm/dB), (3) COSMOS 장비 유형/증폭기 Gain·Tilt 범위, (4) SHB outlier 샘플과 스케일 편차의 일치 여부.

아래 결과 CSV는 [LR-controlled 실험](https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/benchmarks/lr_control)에서 **가중치가 선택된 이후** 만든 검증 전용 진단이며, 원본 공개 TestGround는 사용하지 않는다.

In [ ]:
if lr_control is None:
    print("LR-controlled run 완료 후 아래 진단 파일을 확인할 수 있습니다.")
else:
    for model in ("pi-edfa","transformer"):
        print("\n===== "+model+" =====")
        rows=fetch_csv(f"benchmarks/lr_control/{model}/transfer_validation_rows.csv")
        worst=fetch_csv(f"benchmarks/lr_control/{model}/transfer_worst_channels.csv")
        z=fetch_csv(f"benchmarks/lr_control/{model}/transfer_feature_domain_shift.csv")
        display(Markdown("**최대 절대오차 샘플 TOP 5**"))
        display(rows.head(5))
        display(Markdown("**최악 채널 TOP 5 (ID 및 실측/예측 Gain)**"))
        display(worst.head(5))
        display(Markdown("**COSMOS 스케일러 기준 Target 특성의 분포**"))
        display(z.sort_values("max_abs_z",ascending=False))


## 7. 손실함수와 공식 점수의 불일치

**분류: CONFIRMED CODE DIFFERENCE / IMPACT UNPROVEN**

PI-EDFA와 Transformer 모두 학습의 differentiable surrogate에 **최대오차 페널티 계수 0.04**를 사용한다. 반면 `official_score`에는 **0.10**을 사용한다.

- 학습: `0.04 * relu(active.max() - p95 - 0.7)`
- 평가: `0.10 * max(0, Emax - P95 - 0.7)`

따라서 극단 오차를 줄이는 학습 압력이 공식 평가보다 작게 구현되어 있다는 사실은 확정이다. 단, 이 차이를 29 dB 오차의 단독 원인으로 단정할 수 없고, 추가 학습률/데이터 분포 대조가 먼저 필요하다. 동일 조건에서 손실 가중치 `0.04 vs 0.10` ablation을 해야 한다.

## 8. 원인 판정표 및 개선 우선순위

| 구분 | 증거 | 판정 | 후속 조치 |
|---|---|---|---|
| 체크포인트 점수 보고 불일치 | 로그 최소점과 JSON 최종 점수 상이 | **확정 오류, 수정됨** | 앞으로 모든 stage에 best checkpoint 기준 지표 기록 |
| SHB 큰 오차 | 검증 활성 채널 최대오차 약 29dB | **실제 현상 확인** | worst sample ID / channel별 실측과 예측 확인 |
| 미세조정 LR 너무 낮음 | Transfer LR 1e-4 vs Scratch 2e-3 | **차이 확인, 영향은 대조 중** | 두 모델 LR=2e-3로 통제 |
| Source/Target 분포 이동 | COSMOS scaler를 최근 Train에 재사용 | **가능성** | 글로벌 특성 z-점수, 장비/조건 분포 비교 |
| surrogate 최대오차 가중치 불일치 | 0.04(학습) vs 0.10(점수) | **차이 확인, 영향 미검증** | loss ablation |
| COSMOS 자체의 이득 | source 검증과 target 검증은 서로 다른 모집단 | **미확정** | 전체 COSMOS + 다중 seed / 동일 홀드아웃 |

### 정식 검증 권고

1. **데이터 감사:** SHB worst 샘플의 mask/label/ID/입력 스케일을 먼저 검증한다.
2. **공정한 LR sweep:** PI-EDFA와 Transformer 각각 LR `1e-4, 5e-4, 1e-3, 2e-3`을 동일 분할에 적용한다.
3. **COSMOS 충분히 학습:** 파일 전체 `--cosmos-max-rows 0`, 여러 epoch와 조기 종료를 사용한다. 이 노트북의 소규모 pilot 수치를 최종 COSMOS 성능으로 쓰지 않는다.
4. **Ablation:** source scaler / target normalization 전략, 물리 기반 baseline 유지, max-penalty loss 계수를 한 변수씩 변경한다.
5. **최종 사후평가:** 하이퍼파라미터 확정 이후에만 공개 TestGround에 대해 Aging/SHB/Unseen을 평가한다.

**최종 판정(현재):** 첫 예비 실험에서는 Transfer가 Scratch보다 열세였다. 그러나 학습률·적은 COSMOS epoch·SHB 특이 샘플을 통제하지 않았으므로 'COSMOS 전이학습은 효과 없음'이라는 결론은 아직 정당화되지 않는다.

## 9. 분석 재현·출처 및 변경 이력

- [PI-EDFA 원본 노트북](https://github.com/kimheeseo/LSCNS/blob/main/26OFC_ML_challenge/OFC_2026_Physics_Guided_EDFA_Colab.ipynb)
- [Transformer 원본 코드](https://github.com/kimheeseo/LSCNS/blob/main/26OFC_ML_challenge/PI_EDFA_Transformer/transformer_edfa.py)
- [전이학습 실행 코드](https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/train_transfer.py)
- [공식 공개 데이터](https://github.com/ofc-ml-challenge/ofc-ml-challenge-data-code)
- [최초 실험 JSON](https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/benchmarks/pilot/comparison.json)
- [동일 학습률 대조 JSON](https://github.com/kimheeseo/LSCNS/tree/main/26OFC_ML_challenge/COSMOS_Transfer_Learning/benchmarks/lr_control/comparison.json)
- [원본 GitHub Actions 실행](https://github.com/kimheeseo/LSCNS/actions/runs/41358039001) · [LR 대조실험 실행](https://github.com/kimheeseo/LSCNS/actions/runs/41358641951)

**변경 이력:** 2026-11-02 · 성능 요약치 오류의 원인 확인 및 `train_transfer.py` 수정, 대상 샘플·채널·입력 분포 진단 CSV 저장 기능 추가, LR 통제 실험 구성.

> 이 보고서는 연구 재현을 위한 원인 분석 및 검증 기록이다. 아직 재현되지 않은 가설이나 타 대회 제출자의 비공개 모델 성적을 현재 모델의 실측 결과처럼 표현하지 않는다.